# 论文 4：循环神经网络正则化
## Wojciech Zaremba, Ilya Sutskever, Oriol Vinyals (2014)

### RNN 中的 Dropout

核心观点：只对**非循环连接**应用 Dropout，不要对循环连接应用 Dropout。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## 标准 Dropout

In [ ]:
def dropout(x, dropout_rate=0.5, training=True):
    """
    标准 Dropout。
    训练时：以 dropout_rate 的概率将元素随机置零。
    测试时：按 (1 - dropout_rate) 进行缩放。
    """
    if not training or dropout_rate == 0:
        return x
    
    # 反向 Dropout：在训练阶段完成缩放
    mask = (np.random.rand(*x.shape) > dropout_rate).astype(float)
    return x * mask / (1 - dropout_rate)

# 测试 Dropout
x = np.ones((5, 1))
print("Original:", x.T)
print("With dropout (p=0.5):", dropout(x, 0.5).T)
print("With dropout (p=0.5):", dropout(x, 0.5).T)
print("Test mode:", dropout(x, 0.5, training=False).T)

## 正确应用 Dropout 的 RNN

**关键点**：对**输入**和**输出**应用 Dropout，而不是对循环连接应用！

In [ ]:
class RNNWithDropout:
    def __init__(self, input_size, hidden_size, output_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        
        # 权重
        self.W_xh = np.random.randn(hidden_size, input_size) * 0.01
        self.W_hh = np.random.randn(hidden_size, hidden_size) * 0.01
        self.W_hy = np.random.randn(output_size, hidden_size) * 0.01
        self.bh = np.zeros((hidden_size, 1))
        self.by = np.zeros((output_size, 1))
    
    def forward(self, inputs, dropout_rate=0.0, training=True):
        """
        使用 Dropout 执行前向传播。
        
        应用 Dropout 的位置：
        1. 输入连接（x -> h）
        2. 输出连接（h -> y）
        
        不应用 Dropout 的位置：
        - 循环连接（h -> h）
        """
        h = np.zeros((self.hidden_size, 1))
        outputs = []
        hidden_states = []
        
        for x in inputs:
            # 对输入应用 Dropout
            x_dropped = dropout(x, dropout_rate, training)
            
            # 更新 RNN；循环连接上不使用 Dropout
            h = np.tanh(
                np.dot(self.W_xh, x_dropped) +  # 此处使用 Dropout
                np.dot(self.W_hh, h) +           # 此处不使用 Dropout
                self.bh
            )
            
            # 输出前，对隐藏状态应用 Dropout
            h_dropped = dropout(h, dropout_rate, training)
            
            # 输出
            y = np.dot(self.W_hy, h_dropped) + self.by  # 此处使用 Dropout
            
            outputs.append(y)
            hidden_states.append(h)
        
        return outputs, hidden_states

# 测试
rnn = RNNWithDropout(input_size=10, hidden_size=20, output_size=10)
test_inputs = [np.random.randn(10, 1) for _ in range(5)]

outputs_train, _ = rnn.forward(test_inputs, dropout_rate=0.5, training=True)
outputs_test, _ = rnn.forward(test_inputs, dropout_rate=0.5, training=False)

print(f"Training output[0] mean: {outputs_train[0].mean():.4f}")
print(f"Test output[0] mean: {outputs_test[0].mean():.4f}")

## 变分 Dropout

**关键创新**：在所有时间步使用**同一个** Dropout 掩码！

In [ ]:
class RNNWithVariationalDropout:
    def __init__(self, input_size, hidden_size, output_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        
        # 权重，与前一个模型相同
        self.W_xh = np.random.randn(hidden_size, input_size) * 0.01
        self.W_hh = np.random.randn(hidden_size, hidden_size) * 0.01
        self.W_hy = np.random.randn(output_size, hidden_size) * 0.01
        self.bh = np.zeros((hidden_size, 1))
        self.by = np.zeros((output_size, 1))
    
    def forward(self, inputs, dropout_rate=0.0, training=True):
        """
        变分 Dropout：所有时间步共用同一个掩码。
        """
        h = np.zeros((self.hidden_size, 1))
        outputs = []
        hidden_states = []
        
        # 为整个序列一次性生成掩码
        if training and dropout_rate > 0:
            input_mask = (np.random.rand(self.input_size, 1) > dropout_rate).astype(float) / (1 - dropout_rate)
            hidden_mask = (np.random.rand(self.hidden_size, 1) > dropout_rate).astype(float) / (1 - dropout_rate)
        else:
            input_mask = np.ones((self.input_size, 1))
            hidden_mask = np.ones((self.hidden_size, 1))
        
        for x in inputs:
            # 对每个输入应用同一个掩码
            x_dropped = x * input_mask
            
            # 更新 RNN
            h = np.tanh(
                np.dot(self.W_xh, x_dropped) +
                np.dot(self.W_hh, h) +
                self.bh
            )
            
            # 对每个隐藏状态应用同一个掩码
            h_dropped = h * hidden_mask
            
            # 输出
            y = np.dot(self.W_hy, h_dropped) + self.by
            
            outputs.append(y)
            hidden_states.append(h)
        
        return outputs, hidden_states

# 测试变分 Dropout
var_rnn = RNNWithVariationalDropout(input_size=10, hidden_size=20, output_size=10)
outputs_var, _ = var_rnn.forward(test_inputs, dropout_rate=0.5, training=True)

print("Variational dropout uses consistent masks across timesteps")

## 比较不同的 Dropout 策略

In [ ]:
# 生成合成序列数据
seq_length = 20
test_sequence = [np.random.randn(10, 1) for _ in range(seq_length)]

# 分别使用不同策略运行模型
_, h_no_dropout = rnn.forward(test_sequence, dropout_rate=0.0, training=False)
_, h_standard = rnn.forward(test_sequence, dropout_rate=0.5, training=True)
_, h_variational = var_rnn.forward(test_sequence, dropout_rate=0.5, training=True)

# 转换为数组
h_no_dropout = np.hstack([h.flatten() for h in h_no_dropout]).T
h_standard = np.hstack([h.flatten() for h in h_standard]).T
h_variational = np.hstack([h.flatten() for h in h_variational]).T

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(h_no_dropout, cmap='RdBu', aspect='auto')
axes[0].set_title('No Dropout')
axes[0].set_xlabel('Hidden Unit')
axes[0].set_ylabel('Time Step')

axes[1].imshow(h_standard, cmap='RdBu', aspect='auto')
axes[1].set_title('Standard Dropout (different masks per timestep)')
axes[1].set_xlabel('Hidden Unit')
axes[1].set_ylabel('Time Step')

axes[2].imshow(h_variational, cmap='RdBu', aspect='auto')
axes[2].set_title('Variational Dropout (same mask all timesteps)')
axes[2].set_xlabel('Hidden Unit')
axes[2].set_ylabel('Time Step')

plt.tight_layout()
plt.show()

print("Variational dropout shows consistent patterns (same units dropped throughout)")

## Dropout 的使用位置很重要

In [ ]:
# 可视化 Dropout 的应用位置
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 绘制一个简单的 RNN 示意图
def draw_rnn_cell(ax, title, show_input_dropout, show_hidden_dropout, show_recurrent_dropout):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold')
    
    # 绘制方框
    # 输入
    ax.add_patch(plt.Rectangle((1, 2), 1.5, 1, fill=True, color='lightblue', ec='black'))
    ax.text(1.75, 2.5, 'x_t', ha='center', va='center', fontsize=10)
    
    # 当前隐藏状态
    ax.add_patch(plt.Rectangle((4, 4.5), 2, 2, fill=True, color='lightgreen', ec='black'))
    ax.text(5, 5.5, 'h_t', ha='center', va='center', fontsize=12)
    
    # 前一个隐藏状态
    ax.add_patch(plt.Rectangle((7, 4.5), 2, 2, fill=True, color='lightyellow', ec='black'))
    ax.text(8, 5.5, 'h_{t-1}', ha='center', va='center', fontsize=10)
    
    # 输出
    ax.add_patch(plt.Rectangle((4, 7.5), 2, 1, fill=True, color='lightcoral', ec='black'))
    ax.text(5, 8, 'y_t', ha='center', va='center', fontsize=10)
    
    # 绘制箭头
    # 输入到隐藏状态
    color_input = 'red' if show_input_dropout else 'black'
    width_input = 3 if show_input_dropout else 1
    ax.arrow(2.5, 2.5, 1.3, 2, head_width=0.3, color=color_input, lw=width_input)
    if show_input_dropout:
        ax.text(3.2, 3.5, 'DROPOUT', fontsize=8, color='red', fontweight='bold')
    
    # 循环连接
    color_rec = 'red' if show_recurrent_dropout else 'black'
    width_rec = 3 if show_recurrent_dropout else 1
    ax.arrow(7, 5.5, -0.8, 0, head_width=0.3, color=color_rec, lw=width_rec)
    if show_recurrent_dropout:
        ax.text(6.5, 6.2, 'DROPOUT', fontsize=8, color='red', fontweight='bold')
    
    # 隐藏状态到输出
    color_hidden = 'red' if show_hidden_dropout else 'black'
    width_hidden = 3 if show_hidden_dropout else 1
    ax.arrow(5, 6.6, 0, 0.7, head_width=0.3, color=color_hidden, lw=width_hidden)
    if show_hidden_dropout:
        ax.text(5.5, 7, 'DROPOUT', fontsize=8, color='red', fontweight='bold')

# 错误做法：在所有连接上使用 Dropout
draw_rnn_cell(axes[0, 0], 'WRONG: Dropout Everywhere\n(Disrupts temporal flow)', 
             show_input_dropout=True, show_hidden_dropout=True, show_recurrent_dropout=True)

# 错误做法：只在循环连接上使用 Dropout
draw_rnn_cell(axes[0, 1], 'WRONG: Only Recurrent\n(Loses gradient flow)', 
             show_input_dropout=False, show_hidden_dropout=False, show_recurrent_dropout=True)

# 正确做法：采用 Zaremba 等人的方案
draw_rnn_cell(axes[1, 0], 'CORRECT: Zaremba et al.\n(Input & Output only)', 
             show_input_dropout=True, show_hidden_dropout=True, show_recurrent_dropout=False)

# 不使用 Dropout 的基线
draw_rnn_cell(axes[1, 1], 'Baseline: No Dropout\n(May overfit)', 
             show_input_dropout=False, show_hidden_dropout=False, show_recurrent_dropout=False)

plt.tight_layout()
plt.show()

## 核心要点

### 问题所在
- 在 RNN 中直接套用普通 Dropout 的效果并不好
- 丢弃循环连接会破坏时间信息流
- 标准 Dropout 会在每个时间步更换掩码，因而引入较大噪声

### Zaremba 等人的解决方案

**应当应用 Dropout 的位置：**
- ✅ 输入到隐藏状态的连接（W_xh）
- ✅ 隐藏状态到输出的连接（W_hy）

**不应应用 Dropout 的位置：**
- ❌ 循环连接（W_hh）

### 变分 Dropout
- 所有时间步使用**同一个 Dropout 掩码**
- 比不断更换掩码更加稳定
- 具有更好的理论解释（贝叶斯视角）

### 实验结果
- 显著提升语言建模效果
- 在 Penn Treebank 上，测试困惑度从 78.4 改善到 68.7
- 同样适用于 LSTM 和 GRU

### 实现提示
1. 使用比前馈网络更高的 Dropout 比例（0.5～0.7）
2. 对双向 RNN 的**两个**方向都应用 Dropout
3. 可以堆叠多个 LSTM 层，并在层与层之间应用 Dropout
4. 变分 Dropout：每个序列只生成一次掩码

### 它为什么有效
- 不在循环连接上使用 Dropout，从而保留时间依赖关系
- 对非时间维度的变换进行正则化
- 迫使模型对输入特征缺失更加鲁棒
- 变分方法使用一致的掩码，可以降低方差